In [1]:
import pickle

with open("../../data/game_tags_embeddings.pkl", "rb") as f:
    game_tags_embeddings = pickle.load(f)

with open("../../data/game_tags_semantic_embeddings.pkl", "rb") as f:
    game_tags_semantic_embeddings = pickle.load(f)

print(f"game_tags_embeddings:          {len(game_tags_embeddings)} games")
print(f"game_tags_semantic_embeddings: {len(game_tags_semantic_embeddings)} games")

game_tags_embeddings:          17395 games
game_tags_semantic_embeddings: 25487 games


In [2]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def find_closest_games(target_game, embeddings_dict, top_n=5):
    """
    Finds the most similar games based on their embedding vectors.
    
    :param target_game: Name of the game to search for (str)
    :param embeddings_dict: Dictionary mapping Game Name -> Numpy Array (size 32)
    :param top_n: Number of recommendations to return
    """
    
    # Check if the game exists in our database
    if target_game not in embeddings_dict:
        return f"Error: '{target_game}' not found in the embeddings database."

    # Isolate the target vector and reshape it for sklearn
    # reshape(1, -1) turns it from shape (32,) to (1, 32)
    target_vector = embeddings_dict[target_game].reshape(1, -1)
    
    # Prepare the rest of the data
    # We separate names and vectors so their indexes match up
    game_names = list(embeddings_dict.keys())
    all_vectors = np.array(list(embeddings_dict.values()))
    
    # Calculate Cosine Similarity
    # This compares the target (1, 32) against all games (N, 32) simultaneously
    # It returns an array of scores from -1.0 (opposites) to 1.0 (identical)
    similarity_scores = cosine_similarity(target_vector, all_vectors)[0]
    
    # Sort the results
    # argsort() gives us the indexes from lowest to highest, so we reverse it [::-1]
    ranked_indexes = np.argsort(similarity_scores)[::-1]
    
    # Format the output
    print(f"Games most similar to '{target_game}':\n")
    results = []
    
    for idx in ranked_indexes:
        match_name = game_names[idx]
        score = similarity_scores[idx]
        
        # Skip the target game itself (it will always have a 1.0 score)
        if match_name == target_game:
            continue
            
        results.append((match_name, score))
        print(f"{len(results)}. {match_name} (Similarity Score: {score:.4f})")
        
        # Stop once we hit our desired number of recommendations
        if len(results) == top_n:
            break
            
    return results

In [5]:
import io, contextlib

def compare_embeddings(target_game, top_n=10):
    """Runs find_closest_games on both embedding dicts and prints results side by side."""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        results_base     = find_closest_games(target_game, game_tags_embeddings, top_n=top_n)
        results_semantic = find_closest_games(target_game, game_tags_semantic_embeddings, top_n=top_n)

    if isinstance(results_base, str) or isinstance(results_semantic, str):
        print(results_base if isinstance(results_base, str) else results_semantic)
        return

    max_len = max(len(results_base), len(results_semantic))
    results_base     = list(results_base)     + [("—", float("nan"))] * (max_len - len(results_base))
    results_semantic = list(results_semantic) + [("—", float("nan"))] * (max_len - len(results_semantic))

    print(f"{'=' * 80}")
    print(f"  Comparing recommendations for: '{target_game}'")
    print(f"{'=' * 80}")
    print(f"{'Rank':<6} {'game_tags_embeddings':<38} {'game_tags_semantic_embeddings'}")
    print(f"{'-' * 80}")

    for rank, ((name_b, score_b), (name_s, score_s)) in enumerate(zip(results_base, results_semantic), 1):
        score_b_str = f"{score_b:.4f}" if score_b == score_b else "—"
        score_s_str = f"{score_s:.4f}" if score_s == score_s else "—"
        col_b = f"{name_b} ({score_b_str})"
        col_s = f"{name_s} ({score_s_str})"
        print(f"{rank:<6} {col_b:<38} {col_s}")

compare_embeddings("DOOM Eternal", top_n=10)

  Comparing recommendations for: 'DOOM Eternal'
Rank   game_tags_embeddings                   game_tags_semantic_embeddings
--------------------------------------------------------------------------------
1      Wolfenstein: Youngblood (0.8952)       F.E.A.R. (0.9521)
2      DOOM II (0.8832)                       Final DOOM (0.9518)
3      Wrath: Aeon of Ruin (0.8788)           DOOM II (0.9485)
4      LawBreakers (0.8695)                   Hitman 2 (0.9308)
5      Serious Sam: Siberian Mayhem (0.8669)  Far Cry 2 (0.9246)
6      Turok 2: Seeds of Evil (0.8660)        The Medium (0.9245)
7      Call of Duty: Modern Warfare Trilogy (0.8480) Hitman 2: Silent Assassin (0.9207)
8      Call of Duty: Modern Warfare Remastered (0.8437) Hitman: Blood Money (0.9170)
9      Blood: Fresh Supply (0.8431)           Gynophobia (0.9154)
10     Judge Dredd: Dredd VS Death (0.8397)   Call of Juarez: Bound in Blood (0.9151)
